# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

## Import pandas

We are using the very handy pandas library for dataframes.

In [7]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [10]:
publications = pd.read_csv("database_csv/paper-20200422.csv", header=0)
publications


,pub_date,title,year,file_name,venue,type,volumn,pages,author,excerpt,paper_url
0,2020/1/1,A hybrid deep learning and mechanistic kinetic...,2020.0,2020-CERD-hybrid_fcc,Chemical Engineering Research and Design,J,155(2020),202-210,"Fan Yang, Chaonan Dai, Jianquan Tang, Jin Xuan...",Fluid catalytic cracking (FCC) is one of the m...,https://www.sciencedirect.com/science/article/...
1,2020/1/1,基于GBDT和新型P-GBDT算法的催化裂化装置汽油收率寻优模型的构建与应用,2020.0,2020-SYXB-pgbdt_fcc,石油学报,J,36(1),90-98,王伟，汪坤，杨帆*，戴超男，金继民,催化裂化装置是一个高度非线性和相互强关联的多变量系统，基于数据挖掘技术的分析方法是优化该工艺...,http://www.syxbsyjg.com/CN/10.3969/j.issn.1001...
2,2017/7/1,基于人工智能算法的催化裂化装置汽油收率预测模型的构建与分析,2019.0,2019-SYXB-gbdt_fcc,石油学报,J,35(4),655-665,"杨帆,周敏,戴超男,曹军",基于某石化企业的LIMS(Laboratory information managemen...,http://www.syxbsyjg.com/CN/10.3969/j.issn.1001...
3,2020/5/1,数据智能算法在催化裂化模型分析中的应用研究进展,2020.0,2020-SYXB-review-prev,石油学报,J,36(3),xx-xx,杨帆，周敏，金继民，曹军,催化裂化是一个由多种高度非线性和相互强关联的因素影响的复杂工艺过程，包括原料油性质，反应再...,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [11]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [ ]:
template = \
'''---
title: "{title}"
collection: papers
permalink: /papers/{file}
excerpt: '{abstract}'
date: {date}
venue: {venue}
paperurl: 'http://phanyoung.github.io/files/{file_name}.pdf'
citation: '{AUTHOR}."{title}[type]."<i>venue</i>, {year}, {volum}:p.{pages}.'
---

**专利号**：{SN}   
**申请日**：{date}     
**发明人**:  {AUTHOR} 

核心方法
-----
{excerpt}  

摘要
-----
{abstract}   
  

正文
-----
* [在线阅读]({online_url})
* [论文下载](http://phanyoung.github.io/files/{file_name}.pdf)

Cite it: '{AUTHOR}."{title}[type]."<i>venue</i>, {year}, {volum}:p.{pages}.'
'''

In [ ]:
content_keys = ['title', 'date', 'SN_', 'excerpt', 'SN', 'AUTHOR', 'abstract_zh', 'abstract_en', 'SN_p', 'file']

In [ ]:
def dict_strip(d):
    return {k: d[k].strip() for k in d.keys()}


def gen_md(d, bold_name_list = ['Fan Yang', '杨帆']):
    d = dict_strip(d)
    sn = d['SN']
    sn_p, sn_s = sn.split('.')
    d['SN_p'] = sn_p
    d['SN_'] = sn_p + '-' + sn_s
    filename = d['date'] + '-' + d['SN_']
    d['file'] = filename.strip()
    authors = d['author'].strip()
    d['AUTHOR'] = authors.replace(' ', ',')
    for bold_name in bold_name_list:
        d['AUTHOR'] = replace(bold_name, '<b>'+bold_name+'</b>')
    content = {k: d[k] for k in content_keys}
    return template.format(**content), d['file']

In [5]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"
    
    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [6]:
!ls ../_publications/

2009-10-01-paper-title-number-1.md  2015-10-01-paper-title-number-3.md
2010-10-01-paper-title-number-2.md


In [7]:
!cat ../_publications/2009-10-01-paper-title-number-1.md

---
title: "Paper Title Number 1"
collection: publications
permalink: /publication/2009-10-01-paper-title-number-1
excerpt: 'This paper is about the number 1. The number 2 is left for future work.'
date: 2009-10-01
venue: 'Journal 1'
paperurl: 'http://academicpages.github.io/files/paper1.pdf'
citation: 'Your Name, You. (2009). &quot;Paper Title Number 1.&quot; <i>Journal 1</i>. 1(1).'
---
This paper is about the number 1. The number 2 is left for future work.

[Download paper here](http://academicpages.github.io/files/paper1.pdf)

Recommended citation: Your Name, You. (2009). "Paper Title Number 1." <i>Journal 1</i>. 1(1).